# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 9.30e-14 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.00e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405022e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 6.42061e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797539388882393 - 0.09754482372919099*y - 0.4175945425725998*x + 2.1931519300021916*y^2 - 0.1269608752480836*x*y - 0.8025868707701475*x^2)^2 + (0.9043585861943056 - 0.6454975493868688*y + 0.2742173500241793*x - 0.10770166105288279*y^2 - 1.2293131412594325*x*y - 0.9300170435472996*x^2)^2 + (-0.5739156167209173 - 1.3853512632170424*y - 0.6219655290929101*x - 0.20409240982196922*y^2 - 0.15182435477475986*x*y + 0.4443814867923305*x^2)^2 + (0.5703017871356986 - 0.5184102408490359*y + 0.34367412466167085*x + 0.2555951335437905*y^2 + 0.7618448416641994*x*y - 0.02090567085116562*x^2)^2 + (0.04140298207812761 + 0.04911089940832097*y - 0.49990900774527314*x - 0.24792082949867686*y^2 + 0.2972448439664639*x*y - 0.5054176772471584*x^2)^2 + (-0.25245328219549756 - 0.06336616977802403*y + 0.25394133757830056*x - 0.10018803519680765*y^2 + 0.05960215579665097*x*y - 0.1938133953768724*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Indeed, in that case, SumOfSquares computes the corresponding Gröbner basis which
may take a non-negligible amount of time for large systems of equalities.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  181.119 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  343.544 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
We can use this system in a Sum-of-Squares constraint as follows:

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 1.44e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 1.79e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.64e-01 Pobj:  0.0000000e+00 Ad: 8.07e-01 Dobj:  2.6104524e+00 
Iter:  2 Ap: 7.71e-01 Pobj:  0.0000000e+00 Ad: 7.40e-01 Dobj: -5.9092654e-02 
Iter:  3 Ap: 8.08e-01 Pobj:  0.0000000e+00 Ad: 7.65e-01 Dobj:  5.5549062e-01 
Iter:  4 Ap: 7.56e-01 Pobj:  0.0000000e+00 Ad: 7.79e-01 Dobj:  4.8506833e-01 
Iter:  5 Ap: 6.95e-01 Pobj:  0.0000000e+00 Ad: 6.77e-01 Dobj:  4.3029685e-01 
Iter:  6 Ap: 6.35e-01 Pobj:  0.0000000e+00 Ad: 6.83e-01 Dobj:  2.2137644e-01 
Iter:  7 Ap: 6.21e-01 Pobj:  0.0000000e+00 Ad: 6.43e-01 Dobj:  1.2986387e-01 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 2.94304e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-6.840266128284829e-5 + 0.0006533515365501177*y - 0.00014099245458325368*x + 7.045151423077774*y^2 - 28.23057544339412*x*y + 25.36450308935175*x^2)^2 + (-1.4191759223156591e-5 + 0.000427208968310955*y + 0.0008141386557829616*x + 22.480099296101667*y^2 - 13.865342206716395*x*y - 21.676052197456897*x^2)^2 + (-3.842727619293577e-5 - 0.0007289854656665728*y + 0.0005520063339980325*x + 0.12017756274910303*y^2 + 0.09015765882258095*x*y + 0.06696500252606295*x^2)^2 + (-6.126868743676889e-7 - 0.03996165028579074*y + 0.053400522364902385*x - 0.0002606850520754364*y^2 - 0.00019562859695300265*x*y - 0.0001440006197741722*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

6.35736104123831391999950278797089364690720003636670298874378204345703125e-9 - 2.259369270068523645522250308773811790266140866524671228268207647861535415659131e-7y - 1.62134046154650290051134876333750211275540865384615384615384615384614703022391e-7x - 1.3040110253562334709947524713502531312769860960543155670166015625e-5y² - 1.952804019899952129722849047954724710507434792816638946533203125e-5xy - 7.2995878473850137678848905287143367104363278485834598541259765625e-6x² - 4.638114218617298926498430470625559488932291666666666666666666666666671272623229e-8y³ - 4.991724381990062890205687532822291056315104166666666666666666666666664363688385e-8xy² - 4.17016731204900034202069647459865109340171329677104949951171875e-8x²y + 1.260811516728538733262282151442307692307692307692307692307692307691762417911568e-8y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

7.411877145159650252491593375690770439945254111080430448055267333984375e-9 - 9.546182560241692355093323798755398603186851967341051651881291316104406044779951e-9y - 1.809374530545518120141843190559974083533653846153846153846153846147030223899754e-10x + 7.41186077378615248978377394450944848358631134033203125e-9y² + 4.1372287540308860798177192918956279754638671875e-14xy + 7.4119151350325596094847924177884124219417572021484375e-9x² + 3.8102854205135372467339038848876953125e-13y³ - 2.3447910280083306133747100830078125e-13xy² + 2.1850576903403862161212600767612457275390625e-14x²y + 1.266598176768568989176016587477463942307692307692307692307692307691762417911568e-8y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*